In [5]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
!pip install transformers tqdm

import pandas as pd
from transformers import pipeline
from tqdm import tqdm

In [7]:
# @title
df = pd.read_csv('/content/drive/MyDrive/Int Conference/Data Label/LPDP_cleaned_label.csv')
df = df.dropna(subset=['english_translation', 'normalisasi_text'])

**MODEL DETEKSI EMOSI (Pre-trained)**

In [8]:
# @title
print("Sedang memuat model pelabelan otomatis...")
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=False)

Sedang memuat model pelabelan otomatis...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

**PROSES PELABELAN (Looping)**

In [ ]:
# @title
print("Sedang melabeli data...")

def get_emotion(text):
    try:
        pred = classifier(text[:512])[0]
        return pred['label']
    except:
        return "neutral"

tqdm.pandas()
df['label_emotion'] = df['english_translation'].progress_apply(get_emotion)

Sedang melabeli data...


 27%|██▋       | 39215/144779 [1:00:01<2:07:46, 13.77it/s]

**SIMPAN HASILNYA**

In [ ]:
# @title
df.to_csv('/content/drive/MyDrive/TA DATA PREPROCESSING/Crawling Ulang/DATA_FINAL_WITH_EMOTION.csv', index=False)

print("Selesai! kolom 'label_emotion' ditambahkan.")
print(df[['normalisasi_text', 'label_emotion']].head())
print(df['label_emotion'].value_counts())

**Fasttext + SVM**

In [ ]:
!pip install gensim

In [ ]:
import pandas as pd
import numpy as np
from gensim.models import FastText
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/TA DATA PREPROCESSING/Crawling Ulang/DATA_FINAL_WITH_EMOTION.csv')

print("Jumlah data awal:", len(df))
print("Sebaran awal:", df['label_emotion'].value_counts())

**FILTER HANYA 3 EMOSI**

In [ ]:
target_emotions = ['joy', 'sadness', 'anger']
df = df[df['label_emotion'].isin(target_emotions)].copy()

**UBAH NAMA LABEL KE INDONESIA**

In [ ]:
map_emosi = {
    'joy': 'Senang',
    'sadness': 'Sedih',
    'anger': 'Marah'
}
df['label_emotion'] = df['label_emotion'].map(map_emosi)

print("\nJumlah data setelah filter:", len(df))
print("Sebaran setelah filter:\n", df['label_emotion'].value_counts())

**PENYEIMBANGAN DATA (UNDERSAMPLING)**

In [ ]:
print("\n--- PROSES PENYEIMBANGAN DATA ---")

# 1. Cek jumlah data paling sedikit (Minority Class)
min_count = df['label_emotion'].value_counts().min()
print(f"Jumlah data terkecil (Senang): {min_count}")

# 2. Ambil sampel dari masing-masing kelas sebanyak min_count
df_marah = df[df['label_emotion'] == 'Marah'].sample(n=min_count, random_state=42)
df_sedih = df[df['label_emotion'] == 'Sedih'].sample(n=min_count, random_state=42)
df_senang = df[df['label_emotion'] == 'Senang'].sample(n=min_count, random_state=42)

# 3. Gabungkan kembali
df_balanced = pd.concat([df_marah, df_sedih, df_senang])

# 4. Acak urutannya biar tidak berurutan per kelas
df = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nSebaran Data Setelah Balancing:")
print(df['label_emotion'].value_counts())
print(f"Total Data Baru: {len(df)}")
print("-----------------------------------")

In [ ]:
# ==========================================================
# CEK SEBARAN TAHUN SETELAH BALANCING
# ==========================================================

# 1. Tentukan nama kolom tanggal (Sesuaikan dengan data kamu)
# Biasanya 'created_at' atau 'timestamp' atau 'date'
nama_kolom_tanggal = 'created_at'  # <--- GANTI INI JIKA ERROR

# Cek apakah kolom ada
if nama_kolom_tanggal in df.columns:
    # Konversi ke format datetime agar bisa diambil tahunnya
    df[nama_kolom_tanggal] = pd.to_datetime(df[nama_kolom_tanggal], errors='coerce')

    # Buat kolom baru 'tahun'
    df['tahun'] = df[nama_kolom_tanggal].dt.year

    # 2. Lihat jumlah data per tahun secara global
    print("\n=== Total Data per Tahun ===")
    print(df['tahun'].value_counts().sort_index())

    # 3. Lihat penyebaran Emosi di setiap Tahun (INI YANG PALING PENTING)
    print("\n=== Tabel Silang: Emosi vs Tahun ===")
    crosstab = pd.crosstab(df['tahun'], df['label_emotion'])
    print(crosstab)

    # 4. Visualisasi biar enak dilihat
    plt.figure(figsize=(10, 6))
    sns.heatmap(crosstab, annot=True, fmt='d', cmap='YlGnBu')
    plt.title('Distribusi Data: Tahun vs Emosi')
    plt.ylabel('Tahun')
    plt.xlabel('Emosi')
    plt.show()

else:
    print(f"Kolom '{nama_kolom_tanggal}' tidak ditemukan. Coba cek df.columns")

In [ ]:
sentences = [str(text).split() for text in df['normalisasi_text']]
y = df['label_emotion']

**TRAINING FASTTEXT (FEATURE EXTRACTION)**

In [ ]:
print("\nSedang melatih FastText...")
# vector_size=100 adalah standar
ft_model = FastText(sentences, vector_size=100, window=5, min_count=1, workers=4, epochs=10)

# Fungsi Mengubah Kalimat jadi Vektor (Rata-rata vektor kata)
def text_to_vector(text_list, model):
    vectors = [model.wv[word] for word in text_list if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

# Terapkan vektorisasi ke semua data (Inilah representasi FastText)
X = np.array([text_to_vector(kalimat, ft_model) for kalimat in sentences])

print(f"Shape X (Fitur): {X.shape}")
print(f"Shape y (Label): {y.shape}")

**Normalisasi Fitur (Scaling)**

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Normalisasi Fitur (Angka vektor diubah ke range 0-1)
scaler = MinMaxScaler()
X = scaler.fit_transform(X) # X sekarang sudah dinormalisasi

**SPLIT DATA (80% Latih, 20% Uji)**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

**TRAINING SVM (DENGAN GRID SEARCH)**

In [ ]:
print("\nSedang mencari parameter SVM terbaik...")

# Skenario pengujian parameter SVM
param_grid = {
    'C': [1, 10, 100],            # Parameter Regularisasi
    'kernel': ['rbf', 'linear'],  # Jenis Kernel
    'gamma': ['scale', 'auto']    # Kernel Coefficient
}

svm = SVC()
grid_search = GridSearchCV(svm, param_grid, cv=5, verbose=2, n_jobs=-1)
grid_search.fit(X_train, y_train)

**HASIL & EVALUASI**

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n" + "="*40)
print(f"Akurasi Terbaik: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Parameter Terbaik: {grid_search.best_params_}")
print("="*40)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

**VISUALISASI CONFUSION MATRIX**

In [ ]:
plt.figure(figsize=(10,8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=best_model.classes_,
            yticklabels=best_model.classes_)
plt.title('Confusion Matrix Klasifikasi Emosi (FastText + SVM)')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.show()

**PERCOBAAN KE-2**

**TRAINING FASTTEXT**

In [ ]:
import time
print("\n[1/3] Melatih FastText (Dimensi 300, Epoch 30)...")
start_ft = time.time()

# Upgrade: vector_size=300, epochs=30
ft_model = FastText(sentences, vector_size=300, window=5, min_count=1, workers=4, epochs=30)

def text_to_vector(text_list, model):
    vectors = [model.wv[word] for word in text_list if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X = np.array([text_to_vector(kalimat, ft_model) for kalimat in sentences])

**SCALING DATA**

In [ ]:
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

print(f"FastText Selesai: {time.time()-start_ft:.2f} detik")

**TRAINING SVM (GRID SEARCH)**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("\n[2/3] Grid Search SVM (Fokus RBF)...")
# Parameter Fokus (RBF saja biar cepat tapi akurat)
param_grid = {
    'C': [10, 50, 100, 200],
    'kernel': ['rbf'],
    'gamma': ['scale', 0.1, 1, 10]
}

svm = SVC()
grid_search = GridSearchCV(svm, param_grid, cv=3, verbose=2, n_jobs=-1) # verbose=2 biar kelihatan jalan

start_svm = time.time()
grid_search.fit(X_train, y_train)
print(f"SVM Selesai: {(time.time()-start_svm)/60:.2f} menit")

**EVALUASI**

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n" + "="*40)
print(f"Akurasi Akhir: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Parameter Juara: {grid_search.best_params_}")
print("="*40)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

**Confusion Matrix**

In [ ]:
plt.figure(figsize=(8,6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=best_model.classes_,
            yticklabels=best_model.classes_)
plt.title('Confusion Matrix (Balanced Data)')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.show()

**PERCOBAAN KE-3**

**LOAD FULL DATA (UNTUK MELATIH FASTTEXT)**

In [ ]:
path_file = '/content/drive/MyDrive/TA DATA PREPROCESSING/Crawling Ulang/DATA_FINAL_WITH_EMOTION.csv'
df_full = pd.read_csv(path_file)
df_full = df_full.dropna(subset=['normalisasi_text'])
sentences_full = [str(text).split() for text in df_full['normalisasi_text']]

print(f"Melatih FastText dengan {len(sentences_full)} kalimat (Full Data)...")

**TRAIN FASTTEXT**

In [ ]:
import time
start_ft = time.time()
ft_model_full = FastText(sentences_full, vector_size=300, window=5, min_count=2, workers=4, epochs=20)
print(f"FastText Full Training Selesai: {time.time()-start_ft:.2f} detik")

**DATA BALANCING**

In [ ]:
target_emotions = ['joy', 'sadness', 'anger']
df_labeled = df_full[df_full['label_emotion'].isin(target_emotions)].copy()
map_emosi = {'joy': 'Senang', 'sadness': 'Sedih', 'anger': 'Marah'}
df_labeled['label_emotion'] = df_labeled['label_emotion'].map(map_emosi)

In [ ]:
min_count = df_labeled['label_emotion'].value_counts().min()
df_marah = df_labeled[df_labeled['label_emotion'] == 'Marah'].sample(n=min_count, random_state=42)
df_sedih = df_labeled[df_labeled['label_emotion'] == 'Sedih'].sample(n=min_count, random_state=42)
df_senang = df_labeled[df_labeled['label_emotion'] == 'Senang'].sample(n=min_count, random_state=42)

df_final = pd.concat([df_marah, df_sedih, df_senang])
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nData untuk SVM (Balanced): {len(df_final)} baris")

**VEKTORISASI**

In [ ]:
sentences_svm = [str(text).split() for text in df_final['normalisasi_text']]
y = df_final['label_emotion']

def text_to_vector(text_list, model):
    vectors = [model.wv[word] for word in text_list if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X = np.array([text_to_vector(kalimat, ft_model_full) for kalimat in sentences_svm])

**SCALING**

In [ ]:
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

**TRAINING SVM**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

param_grid = {
    'C': [50, 100],
    'kernel': ['rbf'],
    'gamma': [0.1, 'scale']
}

print("\nRunning SVM Final...")
svm = SVC()
grid_search = GridSearchCV(svm, param_grid, cv=5, verbose=2, n_jobs=-1)
grid_search.fit(X_train, y_train)

**EVALUASI**

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n" + "="*40)
print(f"Akurasi Percobaan Full Data: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("="*40)
print(classification_report(y_test, y_pred))

**CONFUSION MATRIX**

In [ ]:
plt.figure(figsize=(8,6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=best_model.classes_,
            yticklabels=best_model.classes_)
plt.title('Confusion Matrix (Balanced Data)')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.show()

**MAPPING 7 EMOSI KE 3 EMOSI (TIDAK ADA EMOSI YANG DIHAPUS) UNDERSAMPLING**

In [ ]:
# Berdasarkan: Journal of Intelligent Information Systems (2024)

map_7_ke_3 = {
    'anger': 'anger',
    'disgust': 'anger',     # overlap hostile emotion

    'sadness': 'sadness',
    'fear': 'sadness',      # negative passive emotion

    'joy': 'joy',
    'surprise': 'joy',      # positive arousal emotion

    'neutral': None         # dibuang (bukan emosi)
}

df['label_emotion'] = df['label_emotion'].map(map_7_ke_3)

# hapus data yang jadi None (neutral)
df = df.dropna(subset=['label_emotion'])

print("Distribusi setelah remapping 7→3:")
print(df['label_emotion'].value_counts())

In [ ]:
map_emosi = {
    'joy': 'Senang',
    'sadness': 'Sedih',
    'anger': 'Marah'
}
df['label_emotion'] = df['label_emotion'].map(map_emosi)

print("\nJumlah data setelah filter:", len(df))
print("Sebaran setelah filter:\n", df['label_emotion'].value_counts())

**PENYEIMBANGAN DATA (UNDERSAMPLING)**

In [ ]:
print("\n--- PROSES PENYEIMBANGAN DATA ---")

# 1. Cek jumlah data paling sedikit (Minority Class)
min_count = df['label_emotion'].value_counts().min()
print(f"Jumlah data terkecil (Senang): {min_count}")

# 2. Ambil sampel dari masing-masing kelas sebanyak min_count
df_marah = df[df['label_emotion'] == 'Marah'].sample(n=min_count, random_state=42)
df_sedih = df[df['label_emotion'] == 'Sedih'].sample(n=min_count, random_state=42)
df_senang = df[df['label_emotion'] == 'Senang'].sample(n=min_count, random_state=42)

# 3. Gabungkan kembali
df_balanced = pd.concat([df_marah, df_sedih, df_senang])

# 4. Acak urutannya biar tidak berurutan per kelas
df = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nSebaran Data Setelah Balancing:")
print(df['label_emotion'].value_counts())
print(f"Total Data Baru: {len(df)}")
print("-----------------------------------")

In [ ]:
# ==========================================================
# CEK SEBARAN TAHUN SETELAH BALANCING
# ==========================================================

# 1. Tentukan nama kolom tanggal (Sesuaikan dengan data kamu)
# Biasanya 'created_at' atau 'timestamp' atau 'date'
nama_kolom_tanggal = 'created_at'  # <--- GANTI INI JIKA ERROR

# Cek apakah kolom ada
if nama_kolom_tanggal in df.columns:
    # Konversi ke format datetime agar bisa diambil tahunnya
    df[nama_kolom_tanggal] = pd.to_datetime(df[nama_kolom_tanggal], errors='coerce')

    # Buat kolom baru 'tahun'
    df['tahun'] = df[nama_kolom_tanggal].dt.year

    # 2. Lihat jumlah data per tahun secara global
    print("\n=== Total Data per Tahun ===")
    print(df['tahun'].value_counts().sort_index())

    # 3. Lihat penyebaran Emosi di setiap Tahun (INI YANG PALING PENTING)
    print("\n=== Tabel Silang: Emosi vs Tahun ===")
    crosstab = pd.crosstab(df['tahun'], df['label_emotion'])
    print(crosstab)

    # 4. Visualisasi biar enak dilihat
    plt.figure(figsize=(10, 6))
    sns.heatmap(crosstab, annot=True, fmt='d', cmap='YlGnBu')
    plt.title('Distribusi Data: Tahun vs Emosi')
    plt.ylabel('Tahun')
    plt.xlabel('Emosi')
    plt.show()

else:
    print(f"Kolom '{nama_kolom_tanggal}' tidak ditemukan. Coba cek df.columns")

In [ ]:
sentences = [str(text).split() for text in df['normalisasi_text']]
y = df['label_emotion']

**TRAINING FASTTEXT (FEATURE EXTRACTION)**

In [ ]:
print("\nSedang melatih FastText...")
# vector_size=100 adalah standar
ft_model = FastText(sentences, vector_size=100, window=5, min_count=1, workers=4, epochs=10)

# Fungsi Mengubah Kalimat jadi Vektor (Rata-rata vektor kata)
def text_to_vector(text_list, model):
    vectors = [model.wv[word] for word in text_list if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

# Terapkan vektorisasi ke semua data (Inilah representasi FastText)
X = np.array([text_to_vector(kalimat, ft_model) for kalimat in sentences])

print(f"Shape X (Fitur): {X.shape}")
print(f"Shape y (Label): {y.shape}")

**Normalisasi Fitur (Scaling)**

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Normalisasi Fitur (Angka vektor diubah ke range 0-1)
scaler = MinMaxScaler()
X = scaler.fit_transform(X) # X sekarang sudah dinormalisasi

**SPLIT DATA (80% Latih, 20% Uji)**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

**TRAINING SVM (DENGAN GRID SEARCH)**

In [ ]:
print("\nSedang mencari parameter SVM terbaik...")

# Skenario pengujian parameter SVM
param_grid = {
    'C': [1, 10, 100],            # Parameter Regularisasi
    'kernel': ['rbf', 'linear'],  # Jenis Kernel
    'gamma': ['scale', 'auto']    # Kernel Coefficient
}

svm = SVC()
grid_search = GridSearchCV(svm, param_grid, cv=5, verbose=2, n_jobs=-1)
grid_search.fit(X_train, y_train)

**HASIL & EVALUASI**

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n" + "="*40)
print(f"Akurasi Terbaik: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Parameter Terbaik: {grid_search.best_params_}")
print("="*40)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

**VISUALISASI CONFUSION MATRIX**

In [ ]:
plt.figure(figsize=(10,8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=best_model.classes_,
            yticklabels=best_model.classes_)
plt.title('Confusion Matrix Klasifikasi Emosi (FastText + SVM)')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.show()

**PERCOBAAN KE-2**

**TRAINING FASTTEXT**

In [ ]:
import time
print("\n[1/3] Melatih FastText (Dimensi 300, Epoch 30)...")
start_ft = time.time()

# Upgrade: vector_size=300, epochs=30
ft_model = FastText(sentences, vector_size=300, window=5, min_count=1, workers=4, epochs=30)

def text_to_vector(text_list, model):
    vectors = [model.wv[word] for word in text_list if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X = np.array([text_to_vector(kalimat, ft_model) for kalimat in sentences])

**SCALING DATA**

In [ ]:
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

print(f"FastText Selesai: {time.time()-start_ft:.2f} detik")

**TRAINING SVM (GRID SEARCH)**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("\n[2/3] Grid Search SVM (Fokus RBF)...")
# Parameter Fokus (RBF saja biar cepat tapi akurat)
param_grid = {
    'C': [10, 50, 100, 200],
    'kernel': ['rbf'],
    'gamma': ['scale', 0.1, 1, 10]
}

svm = SVC()
grid_search = GridSearchCV(svm, param_grid, cv=3, verbose=2, n_jobs=-1) # verbose=2 biar kelihatan jalan

start_svm = time.time()
grid_search.fit(X_train, y_train)
print(f"SVM Selesai: {(time.time()-start_svm)/60:.2f} menit")

**EVALUASI**

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n" + "="*40)
print(f"Akurasi Akhir: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Parameter Juara: {grid_search.best_params_}")
print("="*40)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

**Confusion Matrix**

In [ ]:
plt.figure(figsize=(8,6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=best_model.classes_,
            yticklabels=best_model.classes_)
plt.title('Confusion Matrix (Balanced Data)')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.show()

**PERCOBAAN KE-3**

**LOAD FULL DATA (UNTUK MELATIH FASTTEXT)**

In [ ]:
path_file = '/content/drive/MyDrive/TA DATA PREPROCESSING/Crawling Ulang/DATA_FINAL_WITH_EMOTION.csv'
df_full = pd.read_csv(path_file)
df_full = df_full.dropna(subset=['normalisasi_text'])
sentences_full = [str(text).split() for text in df_full['normalisasi_text']]

print(f"Melatih FastText dengan {len(sentences_full)} kalimat (Full Data)...")

**TRAIN FASTTEXT**

In [ ]:
import time
start_ft = time.time()
ft_model_full = FastText(sentences_full, vector_size=300, window=5, min_count=2, workers=4, epochs=20)
print(f"FastText Full Training Selesai: {time.time()-start_ft:.2f} detik")

**DATA BALANCING**

In [ ]:
df_labeled = df_full.copy()
df_labeled = df_labeled.dropna(subset=['normalisasi_text'])
df_labeled = df_labeled[df_labeled['normalisasi_text'].str.strip() != ""]
# Berdasarkan: Journal of Intelligent Information Systems (2024)

map_7_ke_3 = {
    'anger': 'Marah',
    'disgust': 'Marah',

    'sadness': 'Sedih',
    'fear': 'Sedih',

    'joy': 'Senang',
    'surprise': 'Senang',

    'neutral': None
}

df_labeled['label_emotion'] = df_labeled['label_emotion'].map(map_7_ke_3)

# hapus neutral
df_labeled = df_labeled.dropna(subset=['label_emotion'])

print("Distribusi setelah remapping 7→3:")
print(df_labeled['label_emotion'].value_counts())

In [ ]:
min_count = df_labeled['label_emotion'].value_counts().min()
df_marah = df_labeled[df_labeled['label_emotion'] == 'Marah'].sample(n=min_count, random_state=42)
df_sedih = df_labeled[df_labeled['label_emotion'] == 'Sedih'].sample(n=min_count, random_state=42)
df_senang = df_labeled[df_labeled['label_emotion'] == 'Senang'].sample(n=min_count, random_state=42)

df_final = pd.concat([df_marah, df_sedih, df_senang])
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nData untuk SVM (Balanced): {len(df_final)} baris")

**VEKTORISASI**

In [ ]:
sentences_svm = [str(text).split() for text in df_final['normalisasi_text']]
y = df_final['label_emotion']

def text_to_vector(text_list, model):
    vectors = [model.wv[word] for word in text_list if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X = np.array([text_to_vector(kalimat, ft_model_full) for kalimat in sentences_svm])

**SCALING**

In [ ]:
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

**TRAINING SVM**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

param_grid = {
    'C': [50, 100],
    'kernel': ['rbf'],
    'gamma': [0.1, 'scale']
}

print("\nRunning SVM Final...")
svm = SVC()
grid_search = GridSearchCV(svm, param_grid, cv=5, verbose=2, n_jobs=-1)
grid_search.fit(X_train, y_train)

**EVALUASI**

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n" + "="*40)
print(f"Akurasi Percobaan Full Data: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("="*40)
print(classification_report(y_test, y_pred))

**CONFUSION MATRIX**

In [ ]:
plt.figure(figsize=(8,6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=best_model.classes_,
            yticklabels=best_model.classes_)
plt.title('Confusion Matrix (Balanced Data)')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.show()

**MAPPING 7 EMOSI KE 3 EMOSI (TIDAK ADA EMOSI YANG DIHAPUS) CLASS WEIGHTING - PENGGANTI UPSAMPLING**

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/TA DATA PREPROCESSING/Crawling Ulang/DATA_FINAL_WITH_EMOTION.csv')

print("Jumlah data awal:", len(df))
print("Sebaran awal:", df['label_emotion'].value_counts())

In [ ]:
# Berdasarkan: Journal of Intelligent Information Systems (2024)

map_7_ke_3 = {
    'anger': 'anger',
    'disgust': 'anger',     # overlap hostile emotion

    'sadness': 'sadness',
    'fear': 'sadness',      # negative passive emotion

    'joy': 'joy',
    'surprise': 'joy',      # positive arousal emotion

    'neutral': None         # dibuang (bukan emosi)
}

df['label_emotion'] = df['label_emotion'].map(map_7_ke_3)

# hapus data yang jadi None (neutral)
df = df.dropna(subset=['label_emotion'])

print("Distribusi setelah remapping 7→3:")
print(df['label_emotion'].value_counts())

In [ ]:
map_emosi = {
    'joy': 'Senang',
    'sadness': 'Sedih',
    'anger': 'Marah'
}
df['label_emotion'] = df['label_emotion'].map(map_emosi)

print("\nJumlah data setelah filter:", len(df))
print("Sebaran setelah filter:\n", df['label_emotion'].value_counts())

**PENYEIMBANGAN DATA (CLASS WEIGHTING - PENGGANTI UPSAMPLING)**

In [ ]:
# @title
from sklearn.utils import resample

print("\n--- PROSES UPSAMPLING DATA ---")

df_marah = df[df['label_emotion'] == 'Marah']
df_sedih = df[df['label_emotion'] == 'Sedih']
df_senang = df[df['label_emotion'] == 'Senang']

max_count = df['label_emotion'].value_counts().max()
print("Jumlah kelas terbesar:", max_count)

df_marah_up = resample(df_marah, replace=True, n_samples=max_count, random_state=42)
df_sedih_up = resample(df_sedih, replace=True, n_samples=max_count, random_state=42)
df_senang_up = resample(df_senang, replace=True, n_samples=max_count, random_state=42)

df = pd.concat([df_marah_up, df_sedih_up, df_senang_up])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nSebaran setelah upsampling:")
print(df['label_emotion'].value_counts())
print("Total:", len(df))

In [ ]:
# CEK SEBARAN TAHUN SETELAH BALANCING

nama_kolom_tanggal = 'created_at'

# Cek apakah kolom ada
if nama_kolom_tanggal in df.columns:
    # Konversi ke format datetime agar bisa diambil tahunnya
    df[nama_kolom_tanggal] = pd.to_datetime(
        df[nama_kolom_tanggal],
        format='%a %b %d %H:%M:%S %z %Y',
        errors='coerce'
    )
    # Buat kolom baru 'tahun'
    df['tahun'] = df[nama_kolom_tanggal].dt.year

    # 2. Lihat jumlah data per tahun secara global
    print("\n=== Total Data per Tahun ===")
    print(df['tahun'].value_counts().sort_index())

    # 3. Lihat penyebaran Emosi di setiap Tahun
    print("\n=== Tabel Silang: Emosi vs Tahun ===")
    crosstab = pd.crosstab(df['tahun'], df['label_emotion'])
    print(crosstab)

    # 4. Visualisasi
    plt.figure(figsize=(10, 6))
    sns.heatmap(crosstab, annot=True, fmt='d', cmap='YlGnBu')
    plt.title('Distribusi Data: Tahun vs Emosi')
    plt.ylabel('Tahun')
    plt.xlabel('Emosi')
    plt.show()

else:
    print(f"Kolom '{nama_kolom_tanggal}' tidak ditemukan. Coba cek df.columns")

In [ ]:
print("Sebelum hapus:", len(df))

df = df[df['tahun'] != 2001]

print("Sesudah hapus:", len(df))
print("\nSebaran tahun baru:")
print(df['tahun'].value_counts().sort_index())

In [ ]:
sentences = [str(text).split() for text in df['normalisasi_text']]
y = df['label_emotion']

**TRAINING FASTTEXT (FEATURE EXTRACTION)**

In [ ]:
print("\nSedang melatih FastText...")
ft_model = FastText(sentences, vector_size=100, window=5, min_count=1, workers=4, epochs=10)

def text_to_vector(text_list, model):
    vectors = [model.wv[word] for word in text_list if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

**Normalisasi Fitur (Scaling)**

In [ ]:
# @title
# from sklearn.preprocessing import MinMaxScaler

# # Normalisasi Fitur (Angka vektor diubah ke range 0-1)
# scaler = MinMaxScaler()
# X = scaler.fit_transform(X) # X sekarang sudah dinormalisasi

In [ ]:
print("\nMembuat representasi fitur global...")

X_full = np.array([text_to_vector(kalimat, ft_model) for kalimat in sentences])

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_full = scaler.fit_transform(X_full)

**SPLIT DATA (80% Latih, 20% Uji)**

In [ ]:
# @title
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
print("\nMembuat subset representatif untuk SVM RBF...")

df_rbf_source = df.copy()
df_rbf_source['strata'] = df_rbf_source['tahun'].astype(str) + "_" + df_rbf_source['label_emotion']

from sklearn.model_selection import train_test_split

max_rbf = 20000
df_rbf, _ = train_test_split(
    df_rbf_source,
    train_size=max_rbf,
    stratify=df_rbf_source['strata'],
    random_state=42
)

df_rbf = df_rbf.reset_index(drop=True)

In [ ]:
sentences_rbf = [str(text).split() for text in df_rbf['normalisasi_text']]
y_rbf = df_rbf['label_emotion']

X_rbf = np.array([text_to_vector(kalimat, ft_model) for kalimat in sentences_rbf])
X_rbf = scaler.transform(X_rbf)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_rbf, y_rbf, test_size=0.2, random_state=42, stratify=y_rbf
)

**TRAINING SVM (DENGAN GRID SEARCH)**

In [ ]:
# @title
# print("\nSedang mencari parameter SVM terbaik...")

# # Skenario pengujian parameter SVM
# param_grid = {
#     'C': [1, 10, 100],            # Parameter Regularisasi
#     'kernel': ['rbf', 'linear'],  # Jenis Kernel
#     'gamma': ['scale', 'auto']    # Kernel Coefficient
# }

# svm = SVC(class_weight='balanced')
# grid_search = GridSearchCV(svm, param_grid, cv=5, verbose=2, n_jobs=-1)
# grid_search.fit(X_train, y_train)

**TRAINING SVM (DENGAN RANDOMIZED SEARCH)**

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform
from sklearn.svm import SVC

param_dist = {
    'C': loguniform(0.1, 100),
    'gamma': loguniform(0.01, 5)
}

svm = SVC(kernel='rbf', class_weight='balanced')


random_search = RandomizedSearchCV(
    svm,
    param_distributions=param_dist,
    n_iter=6,
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

**HASIL & EVALUASI**

In [ ]:
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n" + "="*40)
print(f"Akurasi Terbaik: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Parameter Terbaik: {random_search.best_params_}")
print("="*40)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

**VISUALISASI CONFUSION MATRIX**

In [ ]:
plt.figure(figsize=(10,8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=best_model.classes_,
            yticklabels=best_model.classes_)
plt.title('Confusion Matrix Klasifikasi Emosi (FastText + SVM)')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.show()

**PERCOBAAN KE-2**

**TRAINING FASTTEXT**

In [ ]:
print("\nMembuat subset representatif...")

df_sub = df.copy()
df_sub['strata'] = df_sub['tahun'].astype(str) + "_" + df_sub['label_emotion']

from sklearn.model_selection import train_test_split

max_data = 20000
df_sub, _ = train_test_split(
    df_sub,
    train_size=max_data,
    stratify=df_sub['strata'],
    random_state=42
)

df_sub = df_sub.reset_index(drop=True)

sentences = [str(text).split() for text in df_sub['normalisasi_text']]
y = df_sub['label_emotion']

In [ ]:
import time
print("\n[1/3] Melatih FastText (Dimensi 300, Epoch 30)...")
start_ft = time.time()

# Upgrade: vector_size=300, epochs=30
ft_model = FastText(sentences, vector_size=300, window=5, min_count=1, workers=4, epochs=30)

def text_to_vector(text_list, model):
    vectors = [model.wv[word] for word in text_list if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X = np.array([text_to_vector(kalimat, ft_model) for kalimat in sentences])

**SCALING DATA**

In [ ]:
# @title
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

print(f"FastText Selesai: {time.time()-start_ft:.2f} detik")

**TRAINING SVM (GRID SEARCH)**

In [ ]:
# @title
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# print("\n[2/3] Grid Search SVM (Fokus RBF)...")
# # Parameter Fokus (RBF saja biar cepat tapi akurat)
# param_grid = {
#     'C': [10, 50, 100, 200],
#     'kernel': ['rbf'],
#     'gamma': ['scale', 0.1, 1, 10]
# }

# svm = SVC(class_weight='balanced')
# grid_search = GridSearchCV(svm, param_grid, cv=3, verbose=2, n_jobs=-1) # verbose=2 biar kelihatan jalan

# start_svm = time.time()
# grid_search.fit(X_train, y_train)
# print(f"SVM Selesai: {(time.time()-start_svm)/60:.2f} menit")

**TRAINING SVM (RANDOMIZED SEARCH)**

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
import time

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("\n[2/3] Random Search SVM (RBF Focus)...")

param_dist = {
    'C': loguniform(5, 300),
    'gamma': loguniform(0.01, 5),
    'kernel': ['rbf']
}

svm = SVC(class_weight='balanced')

random_search = RandomizedSearchCV(
    svm,
    param_distributions=param_dist,
    n_iter=12,      # hanya 12 model
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

start_svm = time.time()
random_search.fit(X_train, y_train)
print(f"SVM Selesai: {(time.time()-start_svm)/60:.2f} menit")

**EVALUASI**

In [ ]:
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n" + "="*40)
print(f"Akurasi Akhir: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Parameter Juara: {random_search.best_params_}")
print("="*40)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

**Confusion Matrix**

In [ ]:
plt.figure(figsize=(8,6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=best_model.classes_,
            yticklabels=best_model.classes_)
plt.title('Confusion Matrix (Balanced Data)')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.show()

**PERCOBAAN KE-3**

**LOAD FULL DATA (UNTUK MELATIH FASTTEXT)**

In [ ]:
path_file = '/content/drive/MyDrive/TA DATA PREPROCESSING/Crawling Ulang/DATA_FINAL_WITH_EMOTION.csv'
df_full = pd.read_csv(path_file)
df_full = df_full.dropna(subset=['normalisasi_text'])
sentences_full = [str(text).split() for text in df_full['normalisasi_text']]

print(f"Melatih FastText dengan {len(sentences_full)} kalimat (Full Data)...")

**TRAIN FASTTEXT**

In [ ]:
start_ft = time.time()
ft_model_full = FastText(sentences_full, vector_size=300, window=5, min_count=2, workers=4, epochs=20)
print(f"FastText Full Training Selesai: {time.time()-start_ft:.2f} detik")

**DATA BALANCING**

In [ ]:
df_labeled = df_full.copy()
df_labeled = df_labeled.dropna(subset=['normalisasi_text'])
df_labeled = df_labeled[df_labeled['normalisasi_text'].str.strip() != ""]
# Berdasarkan: Journal of Intelligent Information Systems (2024)

map_7_ke_3 = {
    'anger': 'Marah',
    'disgust': 'Marah',

    'sadness': 'Sedih',
    'fear': 'Sedih',

    'joy': 'Senang',
    'surprise': 'Senang',

    'neutral': None
}

df_labeled['label_emotion'] = df_labeled['label_emotion'].map(map_7_ke_3)

# hapus neutral
df_labeled = df_labeled.dropna(subset=['label_emotion'])

print("Distribusi setelah remapping 7→3:")
print(df_labeled['label_emotion'].value_counts())

In [ ]:
min_count = df_labeled['label_emotion'].value_counts().min()
df_marah = df_labeled[df_labeled['label_emotion'] == 'Marah'].sample(n=min_count, random_state=42)
df_sedih = df_labeled[df_labeled['label_emotion'] == 'Sedih'].sample(n=min_count, random_state=42)
df_senang = df_labeled[df_labeled['label_emotion'] == 'Senang'].sample(n=min_count, random_state=42)

df_final = pd.concat([df_marah, df_sedih, df_senang])
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nData untuk SVM (Balanced): {len(df_final)} baris")

**VEKTORISASI**

In [ ]:
sentences_svm = [str(text).split() for text in df_final['normalisasi_text']]
y = df_final['label_emotion']

def text_to_vector(text_list, model):
    vectors = [model.wv[word] for word in text_list if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X = np.array([text_to_vector(kalimat, ft_model_full) for kalimat in sentences_svm])

**SCALING**

In [ ]:
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

**TRAINING SVM**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

param_grid = {
    'C': [50, 100],
    'kernel': ['rbf'],
    'gamma': [0.1, 'scale']
}

print("\nRunning SVM Final...")
svm = SVC(class_weight='balanced')
grid_search = GridSearchCV(svm, param_grid, cv=5, verbose=2, n_jobs=-1)
grid_search.fit(X_train, y_train)

**EVALUASI**

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n" + "="*40)
print(f"Akurasi Percobaan Full Data: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("="*40)
print(classification_report(y_test, y_pred))

**CONFUSION MATRIX**

In [ ]:
plt.figure(figsize=(8,6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=best_model.classes_,
            yticklabels=best_model.classes_)
plt.title('Confusion Matrix (Balanced Data)')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.show()